## Function Calling (Tool Use)

**Week 09 — Revision Notebook**

এই notebook-এ আছে theory, আমার নিজের **Code**, আমার করা **ভুলগুলো** আর শেষে **Self-check**।
কিছু ভুলে গেলে এখানে এসে পড়ে নিলেই হবে।
প্রতিটা topic-এ একই নিয়ম: মানে → ছোট উদাহরণ → AI-তে কেন লাগে।

### সূচিপত্র (Table of Contents)

- [১. Function Calling কী](#s1)
- [২. get_order_status ও get_inventory function](#s2)
  - [ORDERS: order-এর data](#s3)
  - [get_order_status()](#s4)
  - [INVENTORY: stock-এর data](#s5)
  - [get_inventory()](#s6)
  - [get_inventory(): না পাওয়া আর stock 0](#s7)
- [৩. Tool definition (JSON schema)](#s8)
  - [order-এর tool: `tools` list](#s9)
  - [inventory-র tool: `inventory_tool`](#s10)
- [৪. দুটো tool একসাথে: LLM-এর বাছাই](#s11)
  - [client বানানো + tool লাগে না এমন প্রশ্ন](#s12)
  - [inventory_tool যোগ করে laptop-এর প্রশ্ন](#s13)
- [৫. tool_calls পড়া](#s14)
  - [call, name, args বের করা](#s15)
- [৬. run_tool দিয়ে routing](#s16)
  - [run_tool পরীক্ষা](#s17)
- [৭. Result ফেরত দিয়ে চূড়ান্ত উত্তর](#s18)
  - [laptop: result ফেরত দিয়ে উত্তর](#s19)
  - [পুরো flow আবার: order 102 দিয়ে](#s20)
    - [ধাপ ১: দুই প্রশ্ন একসাথে](#s21)
    - [ধাপ ২: tool_calls পড়া](#s22)
    - [ধাপ ৩: function চালানো](#s23)
    - [ধাপ ৪: messages সাজানো](#s24)
    - [ধাপ ৫: চূড়ান্ত উত্তর](#s25)
- [৮. আমার করা ভুলগুলো](#s26)
- [৯. Self-check](#s27)

<a id="s1"></a>

## ১. Function Calling কী

**মানে:** LLM নিজে কোনো function চালাতে পারে না। সে শুধু বলে দিতে পারে "এই function-টা এই argument দিয়ে চালাও"। Function চালায় **আমার code**, আর result আবার LLM-কে দেখাই।

পুরো flow পাঁচ ধাপে:

1. আমি প্রশ্ন আর `tools`-এর তালিকা (কোন function কী করে, তার বর্ণনা) পাঠাই।
2. LLM ঠিক করে tool লাগবে কিনা। লাগলে উত্তরের বদলে `tool_calls` ফেরত দেয় (function-এর নাম + argument)।
3. আমার code ওই function চালায়।
4. Result-টা `role: "tool"` message হিসেবে LLM-কে ফেরত পাঠাই।
5. LLM result দেখে চূড়ান্ত উত্তর লেখে।

**উদাহরণ:** "order 102 এর অবস্থা কী?"
- LLM বলে: `get_order_status` চালাও, `order_id = 102`।
- আমি চালিয়ে পাই `{'status': 'shipped', 'total': 850}`।
- LLM লেখে: "অর্ডার ১০২ শিপড হয়েছে, মোট ৮৫০"।

**AI-তে কেন লাগে:** LLM আমার order বা stock-এর খবর নিজে জানে না। Function calling দিয়ে সে আমার data আনতে পারে। Agent বানানোর ভিত্তিও এটাই।

<a id="s2"></a>

## ২. get_order_status ও get_inventory function

**মানে:** LLM-কে দেওয়ার মতো দুটো সাধারণ Python function। আগে function দুটো একা ঠিকমতো চলে কিনা দেখি, তারপর LLM-এর সাথে জুড়ি।

**উদাহরণ:** Data আসল database না, ছোট dict। `get_order_status(102)` দিলে order 102-এর খবর আসে, `get_inventory("laptop")` দিলে laptop-এর stock আসে।

**AI-তে কেন লাগে:** Tool আসলে একটা সাধারণ function-ই। Function নিজে ঠিক না চললে LLM-এর সাথে জুড়েও লাভ নেই।

<a id="s3"></a>

### ORDERS: order-এর data

Key হলো order number (সংখ্যা), value হলো আরেকটা dict।

In [10]:
ORDERS = {
    101: {"status": "delivered", "total": 1200},
    102: {"status": "shipped", "total": 850},
    103: {"status": "processing", "total": 430},
}

<a id="s4"></a>

### get_order_status()

`ORDERS.get(order_id, {...})`: key না থাকলে crash না করে দ্বিতীয় মানটা (error dict) ফেরত দেয়।

In [11]:
def get_order_status(order_id):
    order = ORDERS.get(order_id, {"error": "order not found"})
    return order
get_order= get_order_status(101)
print(get_order)

{'status': 'delivered', 'total': 1200}


<a id="s5"></a>

### INVENTORY: stock-এর data

Product-এর নাম থেকে stock-এর সংখ্যা। `mouse`-এর stock **0**, মানে product আছে কিন্তু মজুত নেই।

In [15]:
INVENTORY = {
    "laptop": 12,
    "mouse": 0,
    "keyboard": 5,
}

<a id="s6"></a>

### get_inventory()

`INVENTORY.get(product_name)` key না পেলে `None` দেয়। তাই `if inventory is None` দিয়ে "product নেই" ধরা হয়েছে।
`is None` লাগে, কারণ `0` একটা ঠিক মান। এটা নিয়ে [৮](#s26)-এর ৩ নম্বর ভুল।

In [16]:
def get_inventory(product_name):
    inventory = INVENTORY.get(product_name)
    if inventory is None:
        return {"error": "product not found"}
    return {"product": product_name, "stock": inventory}

result = get_inventory("laptop")
print(result)

{'product': 'laptop', 'stock': 12}


<a id="s7"></a>

### get_inventory(): না পাওয়া আর stock 0

- `monitor` → তালিকায় নেই → `{'error': 'product not found'}`
- `mouse` → আছে, stock 0 → `{'product': 'mouse', 'stock': 0}`

In [ ]:
print(get_inventory("monitor"))
print(get_inventory("mouse"))

{'error': 'product not found'}
{'product': 'mouse', 'stock': 0}


<a id="s8"></a>

## ৩. Tool definition (JSON schema)

**মানে:** LLM আমার Python function চোখে দেখে না। তাকে JSON schema দিয়ে বলে দিতে হয়: function-টার নাম কী, কী করে, কোন argument লাগে।

```
{
  "type": "function",
  "function": {
    "name": ...,          # আমার Python function-এর নাম, হুবহু
    "description": ...,   # কখন এটা লাগবে (LLM এটা পড়েই tool বাছে)
    "parameters": {       # এটাই JSON schema
      "type": "object",
      "properties": { argument-এর নাম: {"type": ..., "description": ...} },
      "required": [ যেগুলো অবশ্যই লাগবে ]
    }
  }
}
```

**উদাহরণ:**
- order tool: নাম `get_order_status`, argument `order_id` (`integer`), `required`।
- inventory tool: নাম `get_inventory`, argument `product_name` (`string`)।

JSON schema-র type-এর নাম `integer`, `string`। Python-এর `int`, `str` না।

**AI-তে কেন লাগে:** LLM tool সম্পর্কে শুধু এই লেখাটুকুই জানে। `name` ভুল হলে আমি function খুঁজে পাব না। `description` ভুল বা ফাঁকা হলে LLM ভুল tool বাছবে।

<a id="s9"></a>

### order-এর tool: `tools` list

প্রথমে শুধু order-এর tool। `tools` একটা list, তাই পরে আরও tool যোগ করা যায়।

In [19]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Returns the current status and total price of an order, given its order id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "integer", "description": "The order's number, for example 101"}
                },
                "required": ["order_id"],
            },
        },
    }
]

<a id="s10"></a>

### inventory-র tool: `inventory_tool`

Inventory-র tool আলাদা variable-এ রাখা। পরে এটা `tools`-এ যোগ হবে।

**লক্ষ্য করো:** এই tool-এর `description` এখনো order-এর কথা বলছে ("Get the status of an order by its order id")। Order tool থেকে কপি করেছিলাম, তাই। LLM এই description পড়েই বাছে, তাই inventory-র জন্য এমন কিছু হলে ভালো হতো: "Get how many units of a product are in stock, given its name"। আমার code অপরিবর্তিত রাখা হয়েছে, এখানে শুধু নোট। দেখো [৮](#s26)-এর ২ আর ৫ নম্বর ভুল।

In [17]:
inventory_tool = {
    "type": "function",
    "function": {
        "name": "get_inventory",
        "description": "Get the status of an order by its order id",
        "parameters": {
            "type": "object",
            "properties": {
                "product_name": {
                    "type": "string",
                    "description": "The name of the product, for example laptop or mouse"
                }
            },
            "required": ["product_name"]
        }
    }
}

<a id="s11"></a>

## ৪. দুটো tool একসাথে: LLM-এর বাছাই

**মানে:** এক request-এ একাধিক tool দিলে LLM প্রশ্ন পড়ে নিজেই বাছে কোনটা লাগবে। বাছে `description` পড়ে। কোনো tool না লাগলে সরাসরি উত্তর দেয়, তখন `tool_calls` হয় `None`।

**উদাহরণ:**
- "তোমার নাম কী?" → tool লাগে না → উত্তর `content`-এ, `tool_calls = None`।
- "laptop কত পিস আছে?" → `get_inventory`, argument `{"product_name": "laptop"}`।

**AI-তে কেন লাগে:** একটা assistant-কে অনেক কাজের tool দিয়ে রাখা যায়। কোনটা কখন লাগবে, সেটা LLM-ই ঠিক করে।

<a id="s12"></a>

### client বানানো + tool লাগে না এমন প্রশ্ন

Groq-এর API key `.env` file থেকে `GROQ_API_KEY` নামে পড়া হয়, code-এ key লেখা নেই। `.env` কখনো commit করা যাবে না (`.gitignore`-এ আছে)।
এখনো `tools`-এ শুধু order-এর tool। প্রশ্নটা এমন যে tool লাগে না। দেখার বিষয় হলো `tool_calls = None`।

In [13]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "তোমার নাম কী?"}],
    tools=tools,
)

message = response.choices[0].message
print(message.content)
print(message.tool_calls)

আমার নাম ChatGPT.
None


<a id="s13"></a>

### inventory_tool যোগ করে laptop-এর প্রশ্ন

`tools.append(inventory_tool)` দিয়ে দ্বিতীয় tool যোগ হলো, এখন list-এ দুটো tool।
এবার `content` হলো `None`, আর `tool_calls`-এ আছে `get_inventory`। LLM নিজেই ঠিক tool বাছল।

In [20]:
tools.append(inventory_tool)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "আমাদের কাছে laptop কত পিস আছে?"}],
    tools=tools,

)

message= response.choices[0].message

print(message.content)
print(message.tool_calls)

None
[ChatCompletionMessageFunctionToolCall(id='fc_e90edb3c-b6e2-4aed-a9de-bcd37ef7e9b5', function=Function(arguments='{"product_name":"laptop"}', name='get_inventory'), type='function')]


<a id="s14"></a>

## ৫. tool_calls পড়া

**মানে:** `message.tool_calls` একটা list। প্রতিটা call-এ তিনটা জিনিস কাজের:
- `call.id`: এই call-এর পরিচয়। Result ফেরত দেওয়ার সময় লাগে।
- `call.function.name`: কোন function চালাতে হবে।
- `call.function.arguments`: argument, কিন্তু **JSON string**, dict না।

তাই `json.loads(...)` দিয়ে dict বানাতে হয়।

**উদাহরণ:** `arguments = '{"product_name":"laptop"}'` (string) → `json.loads` → `{'product_name': 'laptop'}` (dict) → `args["product_name"]`।
`json.load` (শেষে `s` নেই) file থেকে পড়ে। String-এর জন্য `json.loads`, `s` মানে string। দেখো [৮](#s26)-এর ১ নম্বর ভুল।

**AI-তে কেন লাগে:** LLM-এর দেওয়া argument দিয়েই আমার function চলে। ঠিকভাবে না পড়তে পারলে tool চালানোই যায় না।

<a id="s15"></a>

### call, name, args বের করা

এই cell-এ দুটো কাজ আছে:
1. `call`, `name`, `args` বের করা (এই section-এর বিষয়)।
2. `run_tool` function বানানো (পরের section-এর বিষয়)।

শেষের `print(result)` আগের `get_inventory("laptop")`-এর result দেখাচ্ছে। এই cell-এ নতুন করে function চালানো হয়নি।

In [27]:
import json

call = message.tool_calls[0]
name = call.function.name
args = json.loads(call.function.arguments)

def run_tool(name, args):
    if name == "get_order_status":
        return  get_order_status(args["order_id"])
    elif name == "get_inventory":
        return get_inventory(args["product_name"])
    else:
        return {"error": "unknown tool"}

print(result)

{'product': 'laptop', 'stock': 12}


<a id="s16"></a>

## ৬. run_tool দিয়ে routing

**মানে:** Tool অনেক হলে LLM-এর বলা `name` দেখে ঠিক function-এ পাঠাতে হয়। `run_tool(name, args)` এই কাজ করে।
- নাম মিললে সেই function চলে।
- কোনোটাই না মিললে `{"error": "unknown tool"}` ফেরত আসে।

**উদাহরণ:**
- `name = "get_inventory"` → `get_inventory(args["product_name"])` চলে।
- `name = "delete_everything"` → কোনো function চলে না, `{'error': 'unknown tool'}`।

**AI-তে কেন লাগে:** LLM অচেনা বা ভুল নাম বললেও program ভাঙে না। শুধু আমার লিখে দেওয়া function-ই চলে, অন্য কিছু না।

<a id="s17"></a>

### run_tool পরীক্ষা

চেনা tool (`get_inventory`) আর অচেনা tool (`delete_everything`) দুটোই চালিয়ে দেখা।

In [29]:
print(run_tool("get_inventory", {"product_name": "mouse"}))
print(run_tool("delete_everything", {}))

{'product': 'mouse', 'stock': 0}
{'error': 'unknown tool'}


<a id="s18"></a>

## ৭. Result ফেরত দিয়ে চূড়ান্ত উত্তর

**মানে:** Function-এর result LLM-কে ফেরত না দিলে সে জানে না কী পাওয়া গেছে। তাই `messages`-এ তিনটা জিনিস পাঠাই:
1. user-এর আসল প্রশ্ন।
2. LLM-এর আগের message (`message`), যার ভেতরে `tool_calls` আছে।
3. `role: "tool"` message। এতে `tool_call_id = call.id` আর `content = json.dumps(result)`। `content` string হতে হয়, তাই `json.dumps`।

**উদাহরণ:** result `{'product': 'laptop', 'stock': 12}` ফেরত দিলে LLM লেখে "আমাদের কাছে ১২টি ল্যাপটপ রয়েছে।"

**AI-তে কেন লাগে:** Function-এর কাঁচা JSON result LLM সুন্দর বাক্যে বদলে দেয়। `tool_call_id` না মিললে LLM বুঝবে না কোন call-এর result এটা।

<a id="s19"></a>

### laptop: result ফেরত দিয়ে উত্তর

`result` এখনো laptop-এর stock, আর `message`, `call` laptop-এর call থেকে এসেছে। তাই তিনটা অংশ মিলে যায়।

In [28]:
messages = [
    {"role": "user", "content": "আমাদের কাছে laptop কত পিস আছে?"},
    message,
    {
        "role": "tool",
        "tool_call_id": call.id,
        "content": json.dumps(result),
    },
]

final = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=tools,
)

print(final.choices[0].message.content)

আমাদের কাছে **১২টি ল্যাপটপ** রয়েছে।


<a id="s20"></a>

### পুরো flow আবার: order 102 দিয়ে

এবার order-এর প্রশ্ন নিয়ে পাঁচ ধাপে একই flow। ক্রম গুরুত্বপূর্ণ:
- ধাপ ১ চালালে `message` বদলে যায়, তাই ধাপ ২ থেকে ক্রমে চালাতে হবে।
- ধাপ ৩ চালালে `result` এখন order-এর হয়ে যায়। উপরের laptop-এর cell আবার চালাতে চাইলে আগে `get_inventory("laptop")` চালিয়ে `result` ঠিক করে নিতে হবে। এটা নিয়ে [৮](#s26)-এর ৬ নম্বর ভুল।

<a id="s21"></a>

#### ধাপ ১: দুই প্রশ্ন একসাথে

একই প্রশ্নে order আর laptop দুটোই জিজ্ঞেস করা হয়েছে, আর `tools`-এ দুটো tool-ই আছে।
`for call in message.tool_calls` দিয়ে সব call ঘুরে দেখা হয়। এই run-এ LLM মোটে একটা call করেছে: `get_order_status`, `order_id = 102`।
পরের ধাপগুলোর code শুধু প্রথম call (`[0]`) সামলায়। তাই laptop-এর অংশের উত্তর এখানে আসবে না। সব call সামলাতে হলে এই `for` loop-ই লাগবে।

In [30]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "order 102 এর অবস্থা কী, আর laptop কত পিস আছে?"}],
    tools=tools,
)

message = response.choices[0].message

print(len(message.tool_calls))

for call in message.tool_calls:
    print(call.function.name, call.function.arguments)

1
get_order_status {"order_id":102}


<a id="s22"></a>

#### ধাপ ২: tool_calls পড়া

`message`-এর প্রথম call থেকে `name` আর `args` বের করা।

নিচের output আগের এক run-এর। তখন `message`-এ laptop-এর call ছিল। ধাপ ১-এর পরে চালালে `get_order_status` আর `{'order_id': 102}` আসার কথা।

In [23]:
import json

call = message.tool_calls[0]
name = call.function.name
args = json.loads(call.function.arguments)
print(name)
print(args)

get_inventory
{'product_name': 'laptop'}


<a id="s23"></a>

#### ধাপ ৩: function চালানো

`args["order_id"]` দিয়ে সরাসরি function ডাকা হয়েছে। `run_tool("get_order_status", args)` লিখলেও একই result আসত।

In [ ]:
result = get_order_status(order_id=args["order_id"])
print(result)

{'status': 'shipped', 'total': 850}


<a id="s24"></a>

#### ধাপ ৪: messages সাজানো

User-এর প্রশ্ন, LLM-এর আগের `message`, আর `role: "tool"` message। মোট **3** টা।

In [ ]:
messages = [
    {"role": "user", "content": "আমার order 102 এর অবস্থা কী?"},
    message,
    {
        "role": "tool",
        "tool_call_id": call.id,
        "content": json.dumps(result),
    },
]
print(len(messages))

3


<a id="s25"></a>

#### ধাপ ৫: চূড়ান্ত উত্তর

Result সহ `messages` আবার পাঠালে LLM এবার tool না চেয়ে সরাসরি বাক্যে উত্তর দেয়।

In [ ]:
final = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=tools,
)
print(final.choices[0].message.content)

অর্ডার ১০২ টি **শিপড** হয়েছে এবং মোট মূল্য **৳৮৫০**। যদি আরও কোনো তথ্য বা সহায়তা দরকার হয় জানাতে দ্বিধা করবেন না!


<a id="s26"></a>

## ৮. আমার করা ভুলগুলো

| # | আমার ভুল | কেন ভুল | ঠিক নিয়ম |
|---|---|---|---|
| 1 | `json.load(call.function.arguments)` | `load` file খোঁজে, কিন্তু `arguments` একটা string। `AttributeError: 'str' object has no attribute 'read'` | `json.loads(call.function.arguments)`। `s` মানে string |
| 2 | `get_order_status` কপি করে `get_inventory` বানিয়ে নাম না বদলানো (`def`-এর নাম বা schema-র `"name"`) | একই নামে দ্বিতীয় `def` আগেরটাকে ঢেকে দেয়, order-এর function হারিয়ে যায়। Schema-র নাম ভুল হলে LLM যে নাম বলে, `run_tool` সেটা চেনে না | কপির পর সাথে সাথে নাম বদলাও। তিন জায়গায় একই নাম মেলাও: `def`, schema-র `"name"`, `run_tool`-এর `if name == ...` |
| 3 | `if not inventory:` | `0`-কেও "নেই" ধরে। `mouse`-এর stock 0, কিন্তু "product not found" দেখাত | `if inventory is None:`। `None` মানে key নেই, `0` মানে আছে কিন্তু stock শূন্য |
| 4 | `INVENTORY = {"laptop", 12, "mouse", 0}` | `key: value` না থাকলে এটা dict না, **set**। Set-এ `.get` নেই: `AttributeError: 'set' object has no attribute 'get'` | `INVENTORY = {"laptop": 12, "mouse": 0}`। প্রতিটা key-র পর `:` |
| 5 | Schema-র `description`-এ placeholder (`"TODO"`, `"..."`) বা কপি করা পুরোনো লেখা রেখে দেওয়া | LLM description পড়েই tool বাছে। Placeholder বা ভুল লেখা থাকলে সে ভুল tool বাছে বা কোনো tool-ই বাছে না | প্রতিটা tool-এ পরিষ্কার এক বাক্যে লেখো সে কী করে। Argument-এর description-এ একটা উদাহরণ দাও |
| 6 | Notebook restart-এর পর আগের cell না চালিয়ে সরাসরি নিচের cell চালানো | Restart-এ সব variable মুছে যায়। `NameError: name 'tools' is not defined` (বা `client`, `ORDERS`)। আর এই notebook-এ `message`, `args`, `result` বারবার বদলায় | Restart-এর পর উপর থেকে ক্রমে চালাও (Run All)। কোনো cell একা চালানোর আগে ভাবো, সে কোন variable-এর উপর নির্ভর করে |

<a id="s27"></a>

## ৯. Self-check

নিজে উত্তর দিয়ে তারপর নিচে মিলিয়ে নাও।

**Function Calling**
1. Function calling-এ LLM নিজে কি function চালায়? তাহলে কে চালায়?
2. পুরো flow-র পাঁচ ধাপ ক্রমে বলো।

**Function আর Schema**
3. `get_inventory`-তে `if inventory is None` কেন লিখি, `if not inventory` কেন না?
4. Tool definition-এর `name`, `description`, `parameters`, `required` প্রতিটা কী কাজে লাগে?
5. `INVENTORY = {"laptop", 12}` লিখলে কী হয়? কেন?

**LLM-এর বাছাই আর tool_calls পড়া**
6. কোনো tool না লাগলে `content` আর `tool_calls`-এ কী থাকে?
7. `call.function.arguments` কী ধরনের জিনিস? `json.load` আর `json.loads`-এর পার্থক্য কী?
8. `call.id` কেন লাগে?

**Routing আর চূড়ান্ত উত্তর**
9. `run_tool("delete_everything", {})` কী ফেরত দেয়? এই design কেন ভালো?
10. Result ফেরত পাঠানোর `messages`-এ কোন তিনটা জিনিস থাকে? `content`-এ `json.dumps(result)` কেন?
11. একসাথে দুই প্রশ্ন করলে `tool_calls`-এ কয়টা call আসতে পারে? আমার code কয়টা সামলায়?
12. Notebook restart-এর পর `NameError` এলে কী করব?

<details>
<summary><b>উত্তর (আগে নিজে ভাবো, তারপর খোলো)</b></summary>

1. **না।** LLM শুধু বলে কোন function কোন argument দিয়ে চালাতে হবে। চালায় **আমার code**।
2. প্রশ্ন + `tools` পাঠাই → LLM `tool_calls` ফেরত দেয় → আমার code function চালায় → result `role: "tool"` message-এ ফেরত পাঠাই → LLM চূড়ান্ত উত্তর লেখে।
3. `0` একটা ঠিক মান (product আছে, stock নেই)। `if not inventory` লিখলে `0`-ও "নেই" ধরা পড়ে। `is None` শুধু key না থাকলেই true হয়।
4. `name`: আমার function-এর নাম, হুবহু। `description`: কখন এটা লাগবে, LLM এটা পড়েই বাছে। `parameters`: argument-এর নাম, type আর description (JSON schema)। `required`: যেগুলো অবশ্যই দিতে হবে।
5. `key: value` নেই, তাই এটা **set**, dict না। `.get` ডাকলে `AttributeError: 'set' object has no attribute 'get'`।
6. `content`-এ সরাসরি উত্তর থাকে, `tool_calls` হয় `None`।
7. **JSON string**, dict না। `json.load` file থেকে পড়ে, `json.loads` string থেকে। এখানে `json.loads` লাগবে।
8. Result ফেরত দেওয়ার সময় `tool_call_id = call.id` দিতে হয়, যাতে LLM বোঝে এটা কোন call-এর result।
9. `{'error': 'unknown tool'}`। অচেনা নামে কিছু চলে না, program-ও ভাঙে না। শুধু আমার লিখে দেওয়া function-ই চলতে পারে।
10. user-এর প্রশ্ন, LLM-এর আগের `message` (যাতে `tool_calls` আছে), আর `role: "tool"` message। `content` string হতে হয়, তাই dict-কে `json.dumps` দিয়ে string বানাই।
11. এক বা একাধিক আসতে পারে (আমার run-এ ১টা এসেছিল)। আমার code শুধু প্রথমটা (`tool_calls[0]`) সামলায়। সবগুলো সামলাতে `for call in message.tool_calls` loop লাগবে।
12. উপর থেকে ক্রমে cell চালাব (Run All)। কারণ restart-এ সব variable মুছে গেছে।

</details>